# Environment Setup

In [43]:
!pip install albumentations
!pip install pandas
!pip install matplotlib


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [44]:
# Cell 1: Environment Setup
# %matplotlib inline # Uncomment if running in a different environment

!pip -q install einops timm torchmetrics  # already in most fastMRI envs
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from einops import rearrange
import os # For os.makedirs
from skimage.metrics import structural_similarity as ssim   # ← NEW


# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Using device: cuda


# Dataset Class Definition

In [45]:
# Cell 2: Dataset Class Definition (Modified for Augmentation)
import torch
from torch.utils.data import Dataset
import os
from pathlib import Path
import numpy as np # Needed for Albumentations
import albumentations as A
# from albumentations.pytorch import ToTensorV2 # Not strictly needed if data is already tensors and handled manually

class ProcessedFastMRIDataset(Dataset):
    """Dataset for loading preprocessed FastMRI data or creating from raw files"""
    
    def __init__(self, data_dir=None, file_list=None, mode='train', mask_func=None, use_processed=True, no_aug_chance=0.0): # Added no_aug_chance
        self.mode = mode
        self.use_processed = use_processed
        self.no_aug_chance = no_aug_chance # Chance to skip augmentation for a sample
        
        if use_processed:
            self.data_dir = os.path.join(data_dir, mode) # [cite: 1]
            try:
                all_files_in_dir = os.listdir(self.data_dir) # [cite: 1]
            except FileNotFoundError:
                raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

            all_pt_files = sorted([os.path.join(self.data_dir, f) for f in all_files_in_dir if f.endswith('.pt')]) # [cite: 1]

            self.batch_files = [] # [cite: 1]
            for f_path_str in all_pt_files:
                if "metadata" not in Path(f_path_str).name: # [cite: 1]
                    self.batch_files.append(f_path_str) # [cite: 1]
            
            if not self.batch_files: # [cite: 1]
                raise FileNotFoundError(f"No valid .pt files (after filtering 'metadata' files) found in {self.data_dir}")
                
            self.examples = [] # [cite: 1]
            
            for i, batch_file_path in enumerate(self.batch_files): # [cite: 1]
                try:
                    batch_peek = torch.load(batch_file_path, map_location='cpu') # [cite: 1]

                    if not isinstance(batch_peek, dict): # [cite: 1]
                        print(f"  Warning: Skipped {batch_file_path}. Loaded object is not a dictionary (type: {type(batch_peek)}).")
                        continue
                    if 'inputs' not in batch_peek: # [cite: 1]
                        print(f"  Warning: Skipped {batch_file_path}. Missing 'inputs' key. Keys present: {list(batch_peek.keys())}.")
                        continue
                    
                    num_samples = len(batch_peek['inputs']) # [cite: 1]
                    self.examples.extend([(i, j) for j in range(num_samples)]) # [cite: 1]
                    del batch_peek # [cite: 1]

                except Exception as e:
                    print(f"  Error peaking into file {batch_file_path}: {e}. Skipping.")
            
            if not self.examples: # [cite: 1]
                raise ValueError(f"No valid examples could be loaded from {self.data_dir}. Check warnings above.")
            
            print(f"Successfully indexed a total of {len(self.examples)} examples from {len(self.batch_files)} .pt files in {self.data_dir}.")

        else:
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.") # [cite: 1]
        
        # Define augmentation pipelines
        self.TARGET_HEIGHT = 320  # Example, adjust if needed
        self.TARGET_WIDTH = 320   # Example, adjust if needed

        if self.mode == 'train':
            self.geometric_transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=10, p=0.3, border_mode=0, interpolation=1), # Use cv2.INTER_LINEAR for interpolation
                A.RandomScale(scale_limit=0.1, p=0.2, interpolation=1),
                # Add PadIfNeeded and RandomCrop (or CenterCrop) to ensure fixed size
                A.PadIfNeeded(min_height=self.TARGET_HEIGHT, min_width=self.TARGET_WIDTH, border_mode=0, p=1.0),
                A.RandomCrop(height=self.TARGET_HEIGHT, width=self.TARGET_WIDTH, p=1.0),
            ])
            self.intensity_transform = A.Compose([
                 A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
                 # Ensure A.GaussNoise parameters are correct for your albumentations version
                 # If 'var_limit' gives warnings, check help(A.GaussNoise) for your version's API
                 A.GaussNoise(
                    std_range=(0.01, 0.05),         # Choose a reasonable std dev range as a fraction of max value
                    mean_range=(0.0, 0.0),          # Zero mean by default
                    per_channel=True,
                    noise_scale_factor=1.0,
                    p=0.2
                )
            ])
        else: # For 'val' or 'test' mode
            # Ensure validation/test data is also consistently sized, typically via CenterCrop
            self.geometric_transform = A.Compose([
                A.PadIfNeeded(min_height=self.TARGET_HEIGHT, min_width=self.TARGET_WIDTH, border_mode=0, p=1.0),
                A.CenterCrop(height=self.TARGET_HEIGHT, width=self.TARGET_WIDTH, p=1.0),
            ]) # Or set to None if your preprocessed val data is already correctly sized
            self.intensity_transform = None

    def __len__(self):
        return len(self.examples) # [cite: 1]
    
    def __getitem__(self, idx):
        if self.use_processed: # [cite: 1]
            batch_idx, sample_idx = self.examples[idx] # [cite: 1]
            batch_data = torch.load(self.batch_files[batch_idx], map_location='cpu') # [cite: 1]
            inputs_tensor = batch_data['inputs'][sample_idx] # [cite: 1]
            targets_tensor = batch_data['targets'][sample_idx] # [cite: 1]
            del batch_data # [cite: 1]
        else:
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.") # [cite: 1]
        
        # Ensure tensors are 2D [H, W] before converting to NumPy
        if inputs_tensor.ndim > 2: inputs_tensor = inputs_tensor.squeeze()
        if targets_tensor.ndim > 2: targets_tensor = targets_tensor.squeeze()

        inputs_np = inputs_tensor.numpy()
        targets_np = targets_tensor.numpy()

        # Apply augmentations only in 'train' mode and not skipped by chance
        if self.mode == 'train' and random.random() > self.no_aug_chance:
            if self.geometric_transform:
                # Albumentations expects 'image' and 'mask' keys
                augmented = self.geometric_transform(image=inputs_np, mask=targets_np)
                inputs_np = augmented['image']
                targets_np = augmented['mask']
            
            if self.intensity_transform:
                # Intensity transforms only on the input
                augmented_input = self.intensity_transform(image=inputs_np)
                inputs_np = augmented_input['image']

        # Convert back to Tensors
        # The model expects [C, H, W], so add channel dimension if not present
        inputs = torch.from_numpy(inputs_np).unsqueeze(0) # Add channel dim: [1, H, W]
        targets = torch.from_numpy(targets_np).unsqueeze(0) # Add channel dim: [1, H, W]
        
        return inputs, targets

# Loss Functions and Metrics

In [57]:
# Cell 3: Loss Functions and Metrics
class SSIMLoss(nn.Module):
    """SSIM loss module for MRI reconstruction"""
    def __init__(self, win_size=7, k1=0.01, k2=0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer('w', torch.ones(1, 1, win_size, win_size) / win_size**2)
        self.cov_norm = win_size**2 / (win_size**2 - 1)
    
    def forward(self, x, y): # Expects x, y to be 4D tensors [B, C, H, W]
        data_range = 1.0  # Images are normalized to [0,1]
        C1 = (self.k1 * data_range)**2
        C2 = (self.k2 * data_range)**2
        
        # Add channel dim if not present (e.g., if input is [B, H, W])
        if x.ndim == 3: x = x.unsqueeze(1)
        if y.ndim == 3: y = y.unsqueeze(1)

        ux = F.conv2d(x, self.w)
        uy = F.conv2d(y, self.w)
        
        uxx = F.conv2d(x * x, self.w)
        uyy = F.conv2d(y * y, self.w)
        uxy = F.conv2d(x * y, self.w)
        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)
        
        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2
        
        D = (A1 * A2) / (B1 * B2)
        return 1 - D.mean()

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()
        self.ssim_loss = SSIMLoss()
        
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        return self.alpha * l1 + (1 - self.alpha) * ssim

def calculate_psnr_torch(img1, img2): # Renamed to avoid conflict if calculate_psnr means numpy
    """Calculate PSNR between two PyTorch tensors"""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return float('inf') # Perfect match
    return 20 * torch.log10(1 / torch.sqrt(mse))

def calculate_ssim_numpy(img1_np, img2_np): # Explicitly for numpy arrays
    """Calculate SSIM between two numpy arrays"""
    return ssim(img1_np, img2_np, data_range=img1_np.max())

# Common Model Blocks

In [47]:
# %% [code]  Basic conv helpers used by every architecture
class DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU()
        )
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__(); self.pool=nn.AvgPool2d(2); self.conv=DoubleConv(in_ch,out_ch)
    def forward(self,x): return self.conv(self.pool(x))
import torch.nn.functional as F

class Up(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.conv=DoubleConv(in_ch,out_ch)
    # --- Up.forward (final, robust) ------------------------------------------
    def forward(self, x, skip):
        x = self.up(x)
    
        # Spatial size guard (ensure spatial dimensions match for concatenation)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
    
        # The UNet.__init__ method (Cell 140) correctly configures the 
        # 'in_ch' for self.conv (DoubleConv) to be the sum of channels 
        # from the upsampled feature map 'x' and the 'skip' connection.
        # Therefore, no channel trimming is needed here.
        
        concatenated_input = torch.cat([x, skip], dim=1)
        return self.conv(concatenated_input)
    



# Transformer Primitives

In [48]:
# %% [code]  Transformer helpers
class CustomMLP(nn.Sequential):
    def __init__(self,dim,mlp_dim=None,p=0.):
        mlp_dim = mlp_dim or dim*4
        super().__init__(nn.Linear(dim,mlp_dim),nn.GELU(),nn.Dropout(p),
                         nn.Linear(mlp_dim,dim),nn.Dropout(p))
# In Cell 6
class TransformerBlock(nn.Module):
    def __init__(self,dim,heads=8,p=0.):
        super().__init__()
        self.n1=nn.LayerNorm(dim) # Change to n1
        self.attn=nn.MultiheadAttention(dim,heads,dropout=p,batch_first=True)
        self.n2=nn.LayerNorm(dim) # Change to n2
        self.mlp=CustomMLP(dim,p=p)

    def forward(self,x):
        x_res = x
        x = self.n1(x) # Use self.n1
        attn_out, _ = self.attn(x,x,x)
        x = x_res + attn_out

        x_res = x
        x = self.n2(x) # Use self.n2
        mlp_out = self.mlp(x)
        x = x_res + mlp_out
        return x


# BTUNet

In [49]:
# Cell 6: BT-UNet Model Definition (from Transformer Variant Tests (4).ipynb)
class BTUNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, base_channels=32, num_pool_layers=4, 
                 tr_depth=4, tr_heads=8, tr_dropout=0.1): # Added tr_dropout
        super().__init__()
        
        # Define the U-Net core
        # The UNet's 'chans' parameter is BTUNet's 'base_channels'
        self.core_unet = UNet(in_chans=in_chans, out_chans=out_chans, # out_chans for UNet core is not final out_chans
                               chans=base_channels, num_pool_layers=num_pool_layers)

        # Encoder path (re-use from core_unet for clarity and direct access if needed)
        self.inc = self.core_unet.inc
        self.downs = self.core_unet.downs
        
        # Determine channel dimension for TransformerBlock based on UNet's structure
        # After 'num_pool_layers' downsamplings, channels become base_channels * (2^num_pool_layers)
        # The UNet's bottleneck_conv takes this and outputs base_channels * (2^(num_pool_layers+1))
        # This output of bottleneck_conv is what we replace with Transformer blocks.
        # So, C for TransformerBlock is base_channels * (2^num_pool_layers)
        
        # Example: base_channels=32, num_pool_layers=4
        # inc: 32 -> 32
        # d0: 32 -> 64 (H/2)
        # d1: 64 -> 128 (H/4)
        # d2: 128 -> 256 (H/8)
        # d3: 256 -> 512 (H/16) <- This is the input to the Transformer bottleneck
        # So, C = base_channels * (2**num_pool_layers)
        
        C_transformer = base_channels * (2**num_pool_layers)
        
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(dim=C_transformer, heads=tr_heads, p=tr_dropout) for _ in range(tr_depth)]
        )

        # Decoder path (re-use from core_unet)
        # The UNet's Up layers expect specific channel inputs after concatenation.
        # The output of tr_blocks (C_transformer channels) will be the 'x' input to the first Up layer.
        # The skip connection to this first Up layer comes from the last Down layer BEFORE the bottleneck.
        # UNet.ups[0] = Up(in_ch_from_prev_up + in_ch_from_skip, out_ch_current_up)
        # For the first Up layer after Transformer:
        #   in_ch_from_prev_up = C_transformer (output of tr_blocks)
        #   in_ch_from_skip = base_channels * (2**(num_pool_layers-1)) (output of (num_pool_layers-1)-th Down block)
        # This logic is handled within the UNet.ups definition if core_unet.bottleneck_conv is effectively bypased
        # and its output dimensions match C_transformer.

        # Let's adjust how UNet is used:
        # 1. Encoder part up to the layer before original bottleneck_conv
        # 2. Transformer blocks
        # 3. Decoder part, taking output of Transformer and skips from encoder

        # Re-defining UNet parts for BT-UNet structure:
        self.encoder_inc = self.core_unet.inc
        self.encoder_downs = nn.ModuleList() # To store (num_pool_layers) Down blocks
        
        current_ch = base_channels
        for _ in range(num_pool_layers):
            self.encoder_downs.append(Down(current_ch, current_ch * 2))
            current_ch *= 2
        # After this loop, current_ch = base_channels * (2**num_pool_layers) which is C_transformer

        self.transformer_bottleneck = nn.Sequential(
            *[TransformerBlock(dim=C_transformer, heads=tr_heads, p=tr_dropout) for _ in range(tr_depth)]
        )
        
        # Decoder
        self.decoder_ups = nn.ModuleList()
        # Input to first Up layer is C_transformer from Transformer + C_transformer//2 from last encoder skip
        # Output from first Up layer is C_transformer//2
        # So, Up(C_transformer + C_transformer//2, C_transformer//2)

        # current_ch is C_transformer (e.g. 512 if base=32, layers=4)
        for _ in range(num_pool_layers):
            # Skip connection comes from encoder_downs[-(i+1)] or inc
            # Channels for skip: current_ch // 2
            # Channels for x_up (from previous decoder stage or transformer): current_ch
            # So, DoubleConv input for Up module: current_ch + (current_ch // 2)
            # DoubleConv output for Up module: current_ch // 2
            self.decoder_ups.append(Up(current_ch + (current_ch // 2), current_ch // 2))
            current_ch //= 2
            
        self.final_conv = nn.Conv2d(current_ch, out_chans, kernel_size=1) # current_ch should be base_channels

    def forward(self, x):
        # Encoder path
        skip_connections = []
        
        enc_x = self.encoder_inc(x)
        skip_connections.append(enc_x) # H, base_channels
        
        for i, down_layer in enumerate(self.encoder_downs):
            enc_x = down_layer(enc_x)
            if i < len(self.encoder_downs) -1: # Store all but the last output which goes to transformer
                skip_connections.append(enc_x)
        # enc_x is now the input to the transformer bottleneck, shape (B, C_transformer, H_bottleneck, W_bottleneck)

        # Transformer bottleneck
        B, C, H_tr, W_tr = enc_x.shape
        # Reshape for Transformer: (B, C, H, W) -> (B, H*W, C)
        tr_input = rearrange(enc_x, 'b c h w -> b (h w) c')
        tr_output = self.transformer_bottleneck(tr_input)
        # Reshape back: (B, H*W, C) -> (B, C, H, W)
        dec_x = rearrange(tr_output, 'b (h w) c -> b c h w', h=H_tr, w=W_tr)

        # Decoder path
        # Skips are from shallowest to deepest: skip_connections[0] is from inc, skip_connections[-1] is from second to last down
        # We need them in reverse for the Up layers: deepest skip first
        for i, up_layer in enumerate(self.decoder_ups):
            skip = skip_connections[-(i + 1)] # Correct skip connection
            dec_x = up_layer(dec_x, skip)
            
        out = self.final_conv(dec_x)
        
        # Ensure output size matches input size if necessary
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
            
        return out

# Baseline UNet

In [50]:
# Cell 5: UNet Baseline (Core for BT-UNet)
class UNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, chans=32, num_pool_layers=4):
        super().__init__()
        self.in_chans = in_chans
        self.out_chans = out_chans
        self.num_pool_layers = num_pool_layers

        self.inc = DoubleConv(in_chans, chans)
        
        # Store the output channels of each DoubleConv block in the encoder path
        # These will be used for skip connections
        encoder_feature_channels = [chans] 
        
        self.downs = nn.ModuleList()
        current_encoder_ch = chans # Input channels to the first Down module's DoubleConv
        for _ in range(num_pool_layers):
            self.downs.append(Down(current_encoder_ch, current_encoder_ch * 2))
            current_encoder_ch *= 2 # Output channels of this Down module's DoubleConv
            encoder_feature_channels.append(current_encoder_ch) 
        # After loop, current_encoder_ch = chans * 2^num_pool_layers (e.g., 32 * 16 = 512 for chans=32, num_pool_layers=4)
        # encoder_feature_channels = [32, 64, 128, 256, 512]

        # Bottleneck takes the output of the last Down layer
        self.bottleneck_conv = DoubleConv(current_encoder_ch, current_encoder_ch * 2) 
        
        # Decoder
        # current_decoder_ch starts as the output channels of the bottleneck
        current_decoder_ch = current_encoder_ch * 2 # (e.g., 1024)

        self.ups = nn.ModuleList()
        # Skips are taken from encoder_feature_channels in reverse order,
        # excluding the last one (which was input to bottleneck).
        # So, skips are from encoder_feature_channels[num_pool_layers-1], ..., encoder_feature_channels[0]
        # Or, using negative indexing: encoder_feature_channels[-2], [-3], ..., up to the one from self.inc
        
        for i in range(num_pool_layers):
            # Determine the number of channels from the corresponding skip connection in the encoder
            # The skip connection for the i-th Up layer (from deepest, i=0) corresponds to
            # the (num_pool_layers - 1 - i)-th element in encoder_feature_channels list,
            # or more simply, encoder_feature_channels[-(i + 2)] using negative indexing.
            skip_ch_for_this_level = encoder_feature_channels[-(i + 2)] 
            
            # The Up module's DoubleConv takes (channels_from_upsampled_deeper_layer + channels_from_skip)
            # The output channels of this Up stage's DoubleConv is typically skip_ch_for_this_level
            output_ch_of_this_up_stage = skip_ch_for_this_level
            
            # Define the Up module: Up(in_channels_for_DoubleConv, out_channels_for_DoubleConv)
            # in_channels_for_DoubleConv = current_decoder_ch (from upsampled deeper layer) + skip_ch_for_this_level
            self.ups.append(Up(current_decoder_ch + skip_ch_for_this_level, output_ch_of_this_up_stage))
            
            # Update current_decoder_ch for the next (shallower) Up layer
            current_decoder_ch = output_ch_of_this_up_stage 
        
        # The final convolution maps the channels from the last Up layer to out_chans
        # After the loop, current_decoder_ch should be equal to 'chans' (the output of self.inc)
        self.outc = nn.Conv2d(current_decoder_ch, out_chans, kernel_size=1) 

    def forward(self, x):
        # Encoder: Store outputs of each DoubleConv block for skip connections
        encoder_outputs = [] 
        enc_x = self.inc(x)
        encoder_outputs.append(enc_x) 
        for down_module in self.downs:
            enc_x = down_module(enc_x) 
            encoder_outputs.append(enc_x)
        # encoder_outputs list now contains:
        # [output_of_inc, output_of_downs[0].conv, ..., output_of_downs[num_pool_layers-1].conv]

        # Bottleneck: Takes the output of the last Down layer
        b = self.bottleneck_conv(encoder_outputs[-1])

        # Decoder
        dec_x = b
        for i, up_module in enumerate(self.ups):
            # Retrieve the corresponding skip connection from the encoder_outputs list
            # For the i-th Up module (from deepest, i=0), the skip connection is
            # encoder_outputs[num_pool_layers - 1 - i] or encoder_outputs[-(i + 2)]
            skip_connection = encoder_outputs[-(i + 2)] 
            dec_x = up_module(dec_x, skip_connection)
            
        # Final output layer
        out = self.outc(dec_x)
        
        # Ensure output spatial dimensions match input if necessary (though Up and Down should handle this)
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
            
        return out


# Swin-UNet

In [51]:
# Cell 5: Swin-UNet Architecture
from timm.models.swin_transformer import WindowAttention, Mlp as TimmMLP
from timm.models.layers import DropPath

def window_partition(x, window_size: int):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size: int, H: int, W: int):
    num_windows_h = H // window_size
    num_windows_w = W // window_size
    B = windows.shape[0] // (num_windows_h * num_windows_w)
    x = windows.view(B, num_windows_h, num_windows_w, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x
    
class SwinBlock(nn.Module):
    def __init__(self, dim, ws=8, heads=4, drop_path_rate=0.): # Added drop_path_rate
        super().__init__()
        self.dim = dim
        self.window_size = ws
        self.heads = heads

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            num_heads=heads,
            window_size=(ws, ws),
            qkv_bias=True
        )
        self.drop_path1 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = TimmMLP(in_features=dim, hidden_features=int(dim * 4), act_layer=nn.GELU, drop=0.1) # TimmMLP uses 'drop' for dropout
        self.drop_path2 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()


    def forward(self, x): # Expected input x: (B, C, H, W)
        B, C, H, W = x.shape
        if C != self.dim:
            raise ValueError(f"Input channel dimension {C} does not match block dimension {self.dim}")

        x_spatial = rearrange(x, 'b c h w -> b h w c')
        shortcut = x_spatial 

        x_normed = self.norm1(x_spatial)

        H_pad, W_pad = H, W
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x_normed = F.pad(x_normed, (0, 0, pad_l, pad_r, pad_t, pad_b)) 
            H_pad += pad_b
            W_pad += pad_r
        
        x_windows = window_partition(x_normed, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # W-MSA
        # For SW-MSA, cyclic shift and attention mask would be needed.
        attn_windows = self.attn(x_windows, mask=None)
        
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        merged_windows = window_reverse(attn_windows, self.window_size, H_pad, W_pad)

        if pad_r > 0 or pad_b > 0:
            merged_windows = merged_windows[:, :H, :W, :].contiguous()
        
        x_spatial = shortcut + self.drop_path1(merged_windows) # If using DropPath
        # x_spatial = shortcut + merged_windows
        
        x_ffn_input = self.norm2(x_spatial)
        x_ffn = self.mlp(x_ffn_input)
        
        x_out_spatial = x_spatial + self.drop_path2(x_ffn) # If using DropPath
        # x_out_spatial = x_spatial + x_ffn

        x_out = rearrange(x_out_spatial, 'b h w c -> b c h w')
        
        return x_out

class SwinUNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, base=32, ws=8, heads_per_stage=(4,4,4,4),
                 drop_path_rate_max=0.1): # Add max drop_path_rate
        super().__init__()
        chs=[base, base*2, base*4, base*8]
        self.inc=nn.Conv2d(in_chans,chs[0],3,padding=1)

        num_total_swin_blocks = 7 # s1,s2,s3,s4, su3,su2,su1
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate_max, num_total_swin_blocks)]  # Stoch. depth rule

        # Encoder stages
        self.s1=self._stage(chs[0],chs[0],ws, heads=heads_per_stage[0], current_dpr=dpr[0])
        self.d1=nn.Conv2d(chs[0],chs[1],kernel_size=2,stride=2)
        self.s2=self._stage(chs[1],chs[1],ws, heads=heads_per_stage[1], current_dpr=dpr[1])
        self.d2=nn.Conv2d(chs[1],chs[2],kernel_size=2,stride=2)
        self.s3=self._stage(chs[2],chs[2],ws, heads=heads_per_stage[2], current_dpr=dpr[2])
        self.d3=nn.Conv2d(chs[2],chs[3],kernel_size=2,stride=2)
        self.s4=self._stage(chs[3],chs[3],ws, heads=heads_per_stage[3], current_dpr=dpr[3]) # Bottleneck

        # Decoder stages
        self.u3=nn.ConvTranspose2d(chs[3],chs[2],kernel_size=2,stride=2)
        self.su3=self._stage(chs[2]*2,chs[2],ws, heads=heads_per_stage[2], current_dpr=dpr[4])
        self.u2=nn.ConvTranspose2d(chs[2],chs[1],kernel_size=2,stride=2)
        self.su2=self._stage(chs[1]*2,chs[1],ws, heads=heads_per_stage[1], current_dpr=dpr[5])
        self.u1=nn.ConvTranspose2d(chs[1],chs[0],kernel_size=2,stride=2)
        self.su1=self._stage(chs[0]*2,chs[0],ws, heads=heads_per_stage[0], current_dpr=dpr[6])

        self.out=nn.Conv2d(chs[0],out_chans,kernel_size=1)

    def _stage(self,in_ch,out_ch,ws,heads, current_dpr=0.): # Accept current_dpr
        return nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size=3,padding=1,bias=False),
            nn.InstanceNorm2d(out_ch,affine=True),
            nn.GELU(),
            SwinBlock(dim=out_ch, ws=ws, heads=heads, drop_path_rate=current_dpr) # Pass dpr
        )
    def forward(self,x):
        x0 = self.inc(x) # Initial conv
        s1 = self.s1(x0) # Stage 1 + skip
        
        s2_in = self.d1(s1) # Downsample
        s2 = self.s2(s2_in) # Stage 2 + skip
        
        s3_in = self.d2(s2) # Downsample
        s3 = self.s3(s3_in) # Stage 3 + skip
        
        b_in = self.d3(s3)  # Downsample to bottleneck
        b = self.s4(b_in)   # Bottleneck stage
        
        u3_up = self.u3(b)  # Upsample
        u3 = self.su3(torch.cat([u3_up, s3], dim=1)) # Concatenate skip and process
        
        u2_up = self.u2(u3) # Upsample
        u2 = self.su2(torch.cat([u2_up, s2], dim=1)) # Concatenate skip and process
        
        u1_up = self.u1(u2) # Upsample
        u1 = self.su1(torch.cat([u1_up, s1], dim=1)) # Concatenate skip and process
        
        return self.out(u1)

# Generic Trainer

# Visualization Functions

In [52]:
# Cell 6: Training, Validation, and Visualization Functions

def train_epoch(model, dataloader, optimizer, criterion, device, scaler, accumulation_steps=1):
    model.train()
    running_loss = 0.0
    
    with tqdm(dataloader, desc="Training", leave=False) as pbar:
        for i, (inputs, targets) in enumerate(pbar):
            if i % accumulation_steps == 0: 
                optimizer.zero_grad()

            inputs = inputs.to(device)
            targets = targets.to(device)

            if len(inputs.shape) == 3: inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3: targets = targets.unsqueeze(1)

            with autocast(device_type=device.type, enabled=scaler.is_enabled()): 
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                if accumulation_steps > 1:
                    loss = loss / accumulation_steps 

            scaler.scale(loss).backward() 

            if (i + 1) % accumulation_steps == 0: 
                scaler.step(optimizer)
                scaler.update()

            current_loss = loss.item() * (accumulation_steps if accumulation_steps > 1 and (i + 1) % accumulation_steps == 0 else 1)
            running_loss += current_loss
            pbar.set_postfix({'loss': current_loss})
            
    # Handle any remaining steps if dataloader size is not a multiple of accumulation_steps
    if len(dataloader) % accumulation_steps != 0:
        scaler.step(optimizer) 
        scaler.update()
        # optimizer.zero_grad() # Will be zeroed at start of next epoch

    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    all_ssims = [] # Collect all SSIMs for averaging
    
    with torch.no_grad():
        with tqdm(dataloader, desc="Validation", leave=False) as pbar:
            for inputs, targets in pbar:
                inputs = inputs.to(device)
                targets = targets.to(device)
                
                if len(inputs.shape) == 3: inputs = inputs.unsqueeze(1)
                if len(targets.shape) == 3: targets = targets.unsqueeze(1)
                
                # No autocast needed for validation if not training
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                psnr = calculate_psnr_torch(outputs, targets) # Use torch version for consistency on device
                
                running_loss += loss.item()
                running_psnr += psnr.item()
                
                # Calculate SSIM on CPU for each item in batch
                for k in range(outputs.size(0)):
                    output_np = outputs[k, 0].cpu().numpy()
                    target_np = targets[k, 0].cpu().numpy()
                    all_ssims.append(calculate_ssim_numpy(output_np, target_np))
                    
                pbar.set_postfix({'val_loss': loss.item(), 'psnr': psnr.item()})
                
    avg_ssim = np.mean(all_ssims) if all_ssims else 0
    return running_loss / len(dataloader), running_psnr / len(dataloader), avg_ssim

def validate(model, dataloader, criterion, device, data_range_metrics=1.0):
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    all_ssims = [] 
    
    with torch.no_grad():
        with tqdm(dataloader, desc="Validation", leave=False) as pbar:
            for inputs, targets in pbar:
                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                
                if inputs.ndim == 3: inputs = inputs.unsqueeze(1)
                if targets.ndim == 3: targets = targets.unsqueeze(1)
                
                outputs = model(inputs) 
                loss = criterion(outputs, targets)
                
                psnr_batch = calculate_psnr_torch(outputs, targets)
                
                running_loss += loss.item() * inputs.size(0) # Weighted by batch size
                # Accumulate PSNR weighted by batch size for correct averaging
                # Handle inf PSNR by capping or deciding on a large finite value if it affects averaging
                # For now, sum up finite PSNRs and count them
                finite_psnr_mask = ~torch.isinf(psnr_batch)
                if finite_psnr_mask.any():
                     running_psnr += psnr_batch[finite_psnr_mask].sum().item() * inputs.size(0) # This is not quite right. Should be per-image then averaged.
                # Let's average PSNR per batch, then average batch PSNRs. Or sum all PSNRs and divide by total images.
                # Simpler: calculate PSNR for each image in batch, then average.
                
                current_batch_psnr_sum = 0
                num_valid_psnr = 0
                for k_idx in range(outputs.size(0)):
                    psnr_single = calculate_psnr_torch(outputs[k_idx:k_idx+1], targets[k_idx:k_idx+1])
                    if not torch.isinf(psnr_single):
                        current_batch_psnr_sum += psnr_single.item()
                        num_valid_psnr +=1
                if num_valid_psnr > 0:
                    running_psnr += (current_batch_psnr_sum / num_valid_psnr) * inputs.size(0) # Weighted sum of batch average PSNRs


                for k in range(outputs.size(0)):
                    output_np = outputs[k].cpu().squeeze().numpy()
                    target_np = targets[k].cpu().squeeze().numpy()
                    all_ssims.append(calculate_ssim_numpy(output_np, target_np))
                    
                pbar.set_postfix({'val_loss': loss.item(), 
                                  'batch_avg_psnr': (current_batch_psnr_sum / num_valid_psnr if num_valid_psnr > 0 else float('nan'))})
                
    avg_loss = running_loss / len(dataloader.dataset)
    avg_psnr = running_psnr / len(dataloader.dataset) if len(dataloader.dataset) > 0 else 0
    avg_ssim = np.mean(all_ssims) if all_ssims else 0
    return avg_loss, avg_psnr, avg_ssim



def visualize_results(model, dataloader, device, epoch, output_dir_str):
    output_dir = Path(output_dir_str)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    
    inputs, targets = next(iter(dataloader))
    inputs = inputs.to(device)
    targets = targets.to(device)
    
    with torch.no_grad():
        outputs = model(inputs.unsqueeze(1) if len(inputs.shape) == 3 else inputs)
    
    fig, axes = plt.subplots(min(4, inputs.size(0)), 3, figsize=(15, 5 * min(4, inputs.size(0))))
    if min(4, inputs.size(0)) == 1: axes = np.array([axes]) # Ensure axes is 2D for single image
    fig.suptitle(f"SwinUNet Reconstruction - Epoch {epoch}", fontsize=16)
    
    for i in range(min(4, inputs.size(0))): 
        input_img = inputs[i].cpu().squeeze().numpy()
        output_img = outputs[i, 0].cpu().squeeze().numpy()
        target_img = targets[i].cpu().squeeze().numpy()
            
        axes[i, 0].imshow(input_img, cmap='gray')
        axes[i, 0].set_title(f"Input (Undersampled)")
        axes[i, 0].axis('off')
            
        axes[i, 1].imshow(output_img, cmap='gray')
        axes[i, 1].set_title(f"Prediction")
        axes[i, 1].axis('off')
            
        axes[i, 2].imshow(target_img, cmap='gray')
        axes[i, 2].set_title(f"Ground Truth")
        axes[i, 2].axis('off')
            
        psnr_val = calculate_psnr_torch(
            torch.from_numpy(output_img).unsqueeze(0).unsqueeze(0), 
            torch.from_numpy(target_img).unsqueeze(0).unsqueeze(0)
        ).item()
        ssim_val = calculate_ssim_numpy(output_img, target_img)
            
        axes[i, 1].text(
            5, 15, 
            f'PSNR: {psnr_val:.2f}\nSSIM: {ssim_val:.4f}',
            color='white', fontsize=10, 
            bbox=dict(facecolor='black', alpha=0.5)
        )
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_dir / f"epoch_{epoch}.png")
    plt.close(fig)

# run_model definition will be in the next cell (main script)

In [53]:
from pathlib import Path # Ensure Path is imported in this cell or globally

# ... (assuming _orig_init is correctly defined from the non-patched ProcessedFastMRIDataset.__init__)

def _patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)

    good_examples = []
    if hasattr(self, 'batch_files') and self.batch_files: # Check if batch_files exists and is not empty
        for idx, slice_idx in self.examples:
            # Ensure idx is a valid index for self.batch_files
            if 0 <= idx < len(self.batch_files):
                batch_file_string_path = self.batch_files[idx]
                # Convert the string path to a Path object
                path_obj = Path(batch_file_string_path)
                if "metadata" not in path_obj.name:  # Now path_obj.name is correct
                    good_examples.append((idx, slice_idx))
            else:
                print(f"Warning in patch: Invalid index {idx} for batch_files of length {len(self.batch_files)}")
        self.examples = good_examples
    elif not hasattr(self, 'batch_files') or not self.batch_files:
        # This case might occur if _orig_init failed to populate self.batch_files
        # or if use_processed was False (though the error trace suggests use_processed=True)
        print("Warning in patch: self.batch_files not populated or empty, skipping example filtering.")
        # self.examples would remain as whatever _orig_init set it to, or cause error if not set.

# ProcessedFastMRIDataset.__init__ = _patched_init # This line applies the patch
# print("Patched ProcessedFastMRIDataset to ignore files that contain 'metadata'")


# Main Training Script

In [54]:
# Cell 7: Main Hyperparameter Tuning Script for Swin-UNet
import pandas as pd # For results display

def run_tuning_experiment(run_name, model_fn, train_ds, val_ds,
                          batch_size=16, epochs=50, lr=1e-4, frac=1.0,
                          viz_every=10, out_root_str="./runs_swin_overfit", 
                          device_str='cuda', accumulation_steps=1):

    out_dir = Path(out_root_str) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    current_device = torch.device(device_str)

    def make_loader_local(ds, shuffle, fraction_local=1.0): # Renamed to avoid conflict
        if fraction_local < 1.0 and len(ds) > 0 : # Added len(ds) > 0 check
            n = int(len(ds) * fraction_local)
            if n == 0 and len(ds) > 0: n = 1 # Ensure at least one sample if dataset is not empty
            idx = random.sample(range(len(ds)), n) if len(ds) > n else list(range(len(ds)))
            ds_subset = Subset(ds, idx)
        else:
            ds_subset = ds
        
        if len(ds_subset) == 0:
            print(f"Warning: DataLoader for {'train' if shuffle else 'val'} created with 0 samples.")
            # Return an empty loader or handle as an error, depending on desired behavior
            return DataLoader(ds_subset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


        return DataLoader(ds_subset, batch_size=batch_size,
                          shuffle=shuffle, num_workers=2, pin_memory=True, drop_last=(True if shuffle and len(ds_subset) > batch_size else False) )


    train_loader = make_loader_local(train_ds, True,  frac)
    val_loader   = make_loader_local(val_ds,   False, 1.0) # Validate on full validation set

    model     = model_fn().to(current_device)
    criterion = CombinedLoss(alpha=0.84).to(current_device) # As used in pilot
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4) # AdamW from pilot
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)
    scaler = GradScaler(device=current_device.type, enabled=current_device.type=='cuda')


    best_val_loss = float('inf')
    history = [] 

    print(f"\n▶ Starting Run: {run_name} – Training on {len(train_loader.dataset)} slices ({frac*100:.0f}% of train set)")
    print(f"  Model: {model.__class__.__name__}, LR: {lr}, Epochs: {epochs}, Batch: {batch_size}")
    print(f"  Saving to: {out_dir}")


    for ep in range(1, epochs + 1):
        print(f"\n{run_name} | Epoch {ep}/{epochs}")

        tr_loss = train_epoch(model, train_loader, optimizer, criterion, current_device, scaler, accumulation_steps)
        val_loss, val_psnr, val_ssim = validate(model, val_loader, criterion, current_device)
        scheduler.step(val_loss)

        history.append([ep, tr_loss, val_loss, val_psnr, val_ssim])
        print(f"  Train Loss: {tr_loss:.4f}, Val Loss: {val_loss:.4f}, Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}")


        if ep % viz_every == 0 or ep == epochs:
            if len(val_loader.dataset) > 0: # Only visualize if val_loader is not empty
                 visualize_results(model, val_loader, current_device, epoch=ep, output_dir_str=str(out_dir))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({'epoch': ep, 'state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()},
                       out_dir / 'best_model.pth')
            print(f"  Saved new best model at epoch {ep} with val_loss: {best_val_loss:.4f}")

        if ep % 10 == 0: # More frequent checkpointing
            torch.save({'epoch': ep, 'state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()},
                       out_dir / f'ckpt_epoch_{ep}.pth')

    hist_np = np.asarray(history)
    if hist_np.size > 0: # Check if history is not empty
        np.savetxt(out_dir / "history.csv", hist_np, delimiter=',',
                   header="epoch,train_loss,val_loss,psnr,ssim", comments='')

        plt.figure(figsize=(18, 5))
        plt.subplot(1, 3, 1); plt.plot(hist_np[:,0], hist_np[:,1], label='Train Loss'); plt.plot(hist_np[:,0], hist_np[:,2], label='Val Loss')
        plt.title(f"Loss ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
        
        plt.subplot(1, 3, 2); plt.plot(hist_np[:,0], hist_np[:,3]); plt.title(f"PSNR ({run_name})")
        plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)"); plt.grid(True)
        
        plt.subplot(1, 3, 3); plt.plot(hist_np[:,0], hist_np[:,4]); plt.title(f"SSIM ({run_name})")
        plt.xlabel("Epoch"); plt.ylabel("SSIM"); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(out_dir / "training_curves.png")
        plt.close()
    else:
        print(f"Warning: History for {run_name} is empty. No curves plotted.")


    print(f"▶ Finished Run: {run_name}. Best val_loss: {best_val_loss:.4f}")
    return run_name, best_val_loss, (history[-1][3] if history else 0), (history[-1][4] if history else 0) # Return last PSNR/SSIM


# Inference and Model Evaluation

In [55]:
def evaluate_model(model_path, dataloader, device, output_dir="./evaluation"):
    """Evaluate a trained model on a dataset"""
    # Load the model
    model = UNet(in_chans=1, out_chans=1, chans=32, num_pool_layers=4).to(device)

    checkpoint = torch.load(model_info['model_path'], map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize metrics
    psnrs = []
    ssims = []
    
    # Process batches
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(tqdm(dataloader, desc="Evaluating")):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Add channel dimension if needed
            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3:
                targets = targets.unsqueeze(1)
            
            # Make predictions
            outputs = model(inputs)
            
            # Calculate metrics
            for j in range(outputs.size(0)):
                output_np = outputs[j, 0].cpu().numpy()
                target_np = targets[j, 0].cpu().numpy()
                
                psnr = calculate_psnr(outputs[j:j+1], targets[j:j+1],data).item()
                ssim_val = calculate_ssim_numpy(output_np, target_np)
                
                psnrs.append(psnr)
                ssims.append(ssim_val)
            
            # Visualize first batch
            if i == 0:
                fig, axes = plt.subplots(4, 3, figsize=(15, 20))
                fig.suptitle("FastMRI Reconstruction Results", fontsize=16)
                
                for j in range(min(4, outputs.size(0))):
                    # Get images
                    input_img = inputs[j, 0].cpu().numpy()
                    output_img = outputs[j, 0].cpu().numpy()
                    target_img = targets[j, 0].cpu().numpy()
                    
                    # Display input
                    axes[j, 0].imshow(input_img, cmap='gray')
                    axes[j, 0].set_title(f"Input (Undersampled)")
                    axes[j, 0].axis('off')
                    
                    # Display output
                    axes[j, 1].imshow(output_img, cmap='gray')
                    axes[j, 1].set_title(f"Prediction")
                    axes[j, 1].axis('off')
                    
                    # Display target
                    axes[j, 2].imshow(target_img, cmap='gray')
                    axes[j, 2].set_title(f"Ground Truth")
                    axes[j, 2].axis('off')
                    
                    # Add metrics as text
                    axes[j, 1].text(
                        10, 20, 
                        f'PSNR: {psnrs[j]:.2f} dB\nSSIM: {ssims[j]:.4f}',
                        color='white', fontsize=12, 
                        bbox=dict(facecolor='black', alpha=0.5)
                    )
                
                plt.tight_layout()
                plt.subplots_adjust(top=0.95)
                plt.savefig(f"{output_dir}/evaluation_samples.png")
                plt.close()
    
    # Calculate average metrics
    avg_psnr = np.mean(psnrs)
    avg_ssim = np.mean(ssims)
    
    # Print results
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average SSIM: {avg_ssim:.4f}")
    
    # Save metrics
    results = {
        'psnr': psnrs,
        'ssim': ssims,
        'avg_psnr': avg_psnr,
        'avg_ssim': avg_ssim
    }
    
    # Save in text file
    with open(f"{output_dir}/evaluation_results.txt", 'w') as f:
        f.write(f"U-Net Baseline Evaluation Results\n")
        f.write(f"Average PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"Average SSIM: {avg_ssim:.4f}\n")
    
    # Plot histograms of metrics
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(psnrs, bins=20)
    plt.xlabel('PSNR (dB)')
    plt.ylabel('Count')
    plt.title(f'PSNR Histogram (Avg: {avg_psnr:.2f} dB)')
    
    plt.subplot(1, 2, 2)
    plt.hist(ssims, bins=20)
    plt.xlabel('SSIM')
    plt.ylabel('Count')
    plt.title(f'SSIM Histogram (Avg: {avg_ssim:.4f})')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/metric_histograms.png")
    plt.close()
    
    return results

# Example usage (uncomment to run)
# val_dataset = ProcessedFastMRIDataset(data_dir="./processed_fastmri_data", mode='val', use_processed=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
# results = evaluate_model("./unet_output/best_model.pth", val_loader, device)


# Run

In [56]:
# ─── data loaders (put in a cell ABOVE the pilot sweep) ────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("/workspace/fastmri-reconstruction/processed_fastmri_data") # USER: Adjust this path!

try:
    # Use 'val' mode for evaluation. Ensure augmentations are appropriate for validation (minimal or none).
    # The ProcessedFastMRIDataset in Cell 135 should handle 'val' mode correctly based on its init.
    val_dataset = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='val', use_processed=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True) # Adjust batch_size and num_workers if needed
    print(f"Successfully loaded validation dataset with {len(val_dataset)} samples.")
except Exception as e:
    print(f"Error loading validation dataset: {e}")
    raise SystemExit("Validation dataset loading failed. Cannot proceed with evaluation.")

# --- 2. Define Models to Compare ---
# IMPORTANT: You MUST update 'model_path' and 'init_params' for each model accurately.
# 'init_params' must match the parameters used when the model was originally trained and saved.

models_to_compare = [
    {
        'display_name': 'SwinUNet_base64_final',
        'model_type': 'swinunet', # Matches class name (lowercase) or a custom identifier
        'model_path': './runs_swin_final/SwinUNet_FINAL_base64_ws8_lr8e-5_hds8_bs6_acc1/best_model.pth', # USER: Update this path
        'init_params': {'in_chans': 1, 'out_chans': 1, 'base': 64, 'ws': 8, 'heads_per_stage': (8,8,8,8), 'drop_path_rate_max': 0.1} # From SwinUNet.__init__
    },
    {
        'display_name': 'SwinUNet_base80_final',
        'model_type': 'swinunet',
        'model_path': './runs_swin_final/SwinUNet_FINAL_base80_ws8_lr6e-5_hds10_bs4_acc2/best_model.pth', # USER: Update this path
        'init_params': {'in_chans': 1, 'out_chans': 1, 'base': 80, 'ws': 8, 'heads_per_stage': (10,10,10,10), 'drop_path_rate_max': 0.1}
    },
    {
        'display_name': 'BTUNet_bc64_best',
        'model_type': 'btunet',
        'model_path': './runs_bt_unet_experiment/BTUNet_bc64_l4_d4_h8_dr01_lr1e4_bs4_acc2_wd1e4/best_model.pth', # USER: Update this path
        'init_params': {'in_chans': 1, 'out_chans': 1, 'base_channels': 64, 'num_pool_layers': 4, 'tr_depth': 4, 'tr_heads': 8, 'tr_dropout': 0.1}
    },
    {
        'display_name': 'UNet_baseline_best',
        'model_type': 'unet',
        'model_path': './unet_baseline_single_experiment/UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline/best_model.pth', # USER: Update this path
        'init_params': {'in_chans': 1, 'out_chans': 1, 'chans': 32, 'num_pool_layers': 4}
    },
    # Add more dictionaries for other models you have saved
]

# --- 3. Evaluation Loop ---
evaluation_results_list = [] # Renamed to avoid conflict
# The 'validate' function requires a criterion. We can use CombinedLoss or just L1Loss if only focusing on PSNR/SSIM from it.
# Using CombinedLoss as it's defined in your notebook.
criterion_eval = CombinedLoss(alpha=0.84).to(device) # Make sure ssim_data_range matches metric calculation
metrics_data_range = 1.0 # For PSNR calculation consistency

for model_info in models_to_compare:
    print(f"--- Evaluating model: {model_info['display_name']} ---")
    
    model_instance = None
    if model_info['model_type'] == 'swinunet':
        model_instance = SwinUNet(**model_info['init_params']).to(device)
    elif model_info['model_type'] == 'btunet':
        model_instance = BTUNet(**model_info['init_params']).to(device)
    elif model_info['model_type'] == 'unet':
        model_instance = UNet(**model_info['init_params']).to(device)
    else:
        print(f"Unknown model type: {model_info['model_type']} for {model_info['display_name']}. Skipping.")
        continue
        
    try:
        checkpoint = torch.load(model_info['model_path'], map_location=device, weights_only=False)
        # The run_tuning_experiment function saves the model state as 'model_state_dict'
        if 'model_state_dict' in checkpoint:
            model_instance.load_state_dict(checkpoint['model_state_dict'])
        elif 'state_dict' in checkpoint: # Some of your earlier scripts might have used 'state_dict'
             model_instance.load_state_dict(checkpoint['state_dict'])
        elif 'state' in checkpoint: # Your original run_model used 'state'
             model_instance.load_state_dict(checkpoint['state'])
        else: # If the .pth file is just the state_dict itself
            model_instance.load_state_dict(checkpoint)
        print(f"Successfully loaded weights from {model_info['model_path']}")
    except Exception as e:
        print(f"Error loading model {model_info['display_name']} from {model_info['model_path']}: {e}. Skipping.")
        continue
        
    # Use the existing validate function (from Cell 142)
    val_loss, val_psnr, val_ssim = validate(model_instance, val_loader, criterion_eval, device, data_range_metrics=metrics_data_range)
    
    evaluation_results_list.append({
        'Model Name': model_info['display_name'],
        'Validation Loss': val_loss,
        'Validation PSNR (dB)': val_psnr,
        'Validation SSIM': val_ssim,
        'Path': model_info['model_path']
    })
    print(f"Results for {model_info['display_name']}: Val Loss: {val_loss:.4f}, Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}\n")

# --- 4. Display Comparative Results ---
if evaluation_results_list:
    results_df_comparison = pd.DataFrame(evaluation_results_list)
    # Sort by your preferred metric, e.g., Validation Loss or Validation PSNR
    results_df_comparison = results_df_comparison.sort_values(by="Validation PSNR (dB)", ascending=False) 
    print("\n--- Overall Model Comparison ---")
    display(results_df_comparison) # This will display the table in Jupyter
    
    # Save the comparison to a CSV file
    comparison_summary_path = Path("./all_models_comparison_summary.csv") # Choose a suitable path
    results_df_comparison.to_csv(comparison_summary_path, index=False)
    print(f"\nComparison summary saved to: {comparison_summary_path}")
else:
    print("No models were successfully evaluated.")


Successfully indexed a total of 199 examples from 13 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/val.
Successfully loaded validation dataset with 199 samples.
--- Evaluating model: SwinUNet_base64_final ---
Successfully loaded weights from ./runs_swin_final/SwinUNet_FINAL_base64_ws8_lr8e-5_hds8_bs6_acc1/best_model.pth


Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Results for SwinUNet_base64_final: Val Loss: 0.0352, Val PSNR: 62.41, Val SSIM: 0.7274

--- Evaluating model: SwinUNet_base80_final ---
Successfully loaded weights from ./runs_swin_final/SwinUNet_FINAL_base80_ws8_lr6e-5_hds10_bs4_acc2/best_model.pth


Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Results for SwinUNet_base80_final: Val Loss: 0.0356, Val PSNR: 62.31, Val SSIM: 0.7182

--- Evaluating model: BTUNet_bc64_best ---
Error loading model BTUNet_bc64_best from ./runs_bt_unet_experiment/BTUNet_bc64_l4_d4_h8_dr01_lr1e4_bs4_acc2_wd1e4/best_model.pth: Error(s) in loading state_dict for BTUNet:
	Missing key(s) in state_dict: "tr_blocks.0.n1.weight", "tr_blocks.0.n1.bias", "tr_blocks.0.n2.weight", "tr_blocks.0.n2.bias", "tr_blocks.1.n1.weight", "tr_blocks.1.n1.bias", "tr_blocks.1.n2.weight", "tr_blocks.1.n2.bias", "tr_blocks.2.n1.weight", "tr_blocks.2.n1.bias", "tr_blocks.2.n2.weight", "tr_blocks.2.n2.bias", "tr_blocks.3.n1.weight", "tr_blocks.3.n1.bias", "tr_blocks.3.n2.weight", "tr_blocks.3.n2.bias", "transformer_bottleneck.0.n1.weight", "transformer_bottleneck.0.n1.bias", "transformer_bottleneck.0.n2.weight", "transformer_bottleneck.0.n2.bias", "transformer_bottleneck.1.n1.weight", "transformer_bottleneck.1.n1.bias", "transformer_bottleneck.1.n2.weight", "transformer_bottlen

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Results for UNet_baseline_best: Val Loss: 0.0496, Val PSNR: 53.23, Val SSIM: 0.7134


--- Overall Model Comparison ---


,Model Name,Validation Loss,Validation PSNR (dB),Validation SSIM,Path
0,SwinUNet_base64_final,0.035195,62.407065,0.727364,./runs_swin_final/SwinUNet_FINAL_base64_ws8_lr...
1,SwinUNet_base80_final,0.035612,62.311012,0.718193,./runs_swin_final/SwinUNet_FINAL_base80_ws8_lr...
2,UNet_baseline_best,0.049621,53.231098,0.713352,./unet_baseline_single_experiment/UNet_chans32...



Comparison summary saved to: all_models_comparison_summary.csv
